# Extract Skills from Job Descriptions (JD) - Fixed Version

This notebook processes multiple parquet files with correct SkillNER integration.

## ⚠️ Important Prerequisites:

1. **Download Knowledge Base** (First time only):
   ```bash
   skillner-download ESCO_EN
   ```

2. **Verify Installation**:
   ```python
   from skillner.core import Pipeline, Document
   from skillner.text_loaders import StrTextLoader
   from skillner.matchers import SlidingWindowMatcher
   from skillner.conflict_resolvers import SpanProcessor
   ```

## Key Features:
- ✅ Correct SkillNER API usage
- ✅ Batch processing to avoid memory overflow
- ✅ Parallel processing support (optional)
- ✅ Progress tracking
- ✅ Resume capability

## A100 Server Configuration:
- Recommended: BATCH_SIZE = 50
- Recommended: ROWS_PER_CHUNK = 10000
- Recommended: N_WORKERS = 16 (for parallel mode)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc
import sys
import json
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Fix import path for local skillner module
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))
    print(f"✓ Added {parent_dir} to Python path")

# Import SkillNER components (correct way)
try:
    from skillner.core import Pipeline, Document
    from skillner.text_loaders import StrTextLoader
    from skillner.matchers import SlidingWindowMatcher
    from skillner.conflict_resolvers import SpanProcessor
    print("✓ SkillNER imported successfully!")
except ImportError as e:
    print(f"❌ Error importing SkillNER: {e}")
    print("\nPlease make sure:")
    print("  1. SkillNER is installed: pip install skillner")
    print("  2. Knowledge base is downloaded: skillner-download ESCO_EN")
    raise

## Load Knowledge Base

In [ ]:
# Load the skills knowledge base
print("Loading skills knowledge base...")

# Try to find the knowledge base file
# Default location after running: skillner-download ESCO_EN
kb_paths = [
    Path.home() / ".skillner" / "skill_db.json",
    Path.home() / ".config" / "skillner" / "skill_db.json",
    Path("/usr/local/share/skillner/skill_db.json"),
    Path("./skill_db.json"),
]

kb_file = None
for path in kb_paths:
    if path.exists():
        kb_file = path
        break

if kb_file is None:
    print("❌ Knowledge base not found!")
    print("\nPlease download it first:")
    print("  skillner-download ESCO_EN")
    print("\nSearched in:")
    for p in kb_paths:
        print(f"  - {p}")
    raise FileNotFoundError("Knowledge base not found")

print(f"✓ Found knowledge base: {kb_file}")

# Load the knowledge base
with open(kb_file, 'r') as f:
    skill_db = json.load(f)

print(f"✓ Loaded {len(skill_db)} skills from knowledge base")

# Create query function for skills
def query_skills(text: str):
    """Query skills from knowledge base"""
    text_lower = text.lower()
    return skill_db.get(text_lower, [])

print("✓ Knowledge base ready!")

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Input/Output paths
INPUT_DIR = Path("../JD")  # Directory containing parquet files
OUTPUT_DIR = Path("./output/extracted_skills")  # Output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Processing parameters - ADJUST BASED ON YOUR SERVER
BATCH_SIZE = 50  # Number of files to process at once
ROWS_PER_CHUNK = 10000  # Number of rows per chunk

# Parallel processing (set to False if you encounter issues)
USE_PARALLEL = True
N_WORKERS = 16  # Number of parallel workers

# Column names - UPDATE THESE TO MATCH YOUR DATA
TEXT_COLUMN = 'job_description'  # Column containing text
ID_COLUMN = 'job_id'  # ID column (optional)

# SkillNER matcher parameters
MAX_WINDOW_SIZE = 4  # Maximum phrase length to match

# Resume capability
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_FILE = OUTPUT_DIR / "processed_files.txt"

print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Batch size: {BATCH_SIZE} files")
print(f"Chunk size: {ROWS_PER_CHUNK} rows")
print(f"Text column: '{TEXT_COLUMN}'")
print(f"Parallel processing: {USE_PARALLEL}")
if USE_PARALLEL:
    print(f"Number of workers: {N_WORKERS}")
print("="*70)

## Helper Functions

In [ ]:
def get_processed_files():
    """Get set of already processed files"""
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            return set(line.strip() for line in f)
    return set()

def mark_file_as_processed(filename):
    """Mark a file as processed"""
    with open(CHECKPOINT_FILE, 'a') as f:
        f.write(f"{filename}\n")

def extract_skills_from_text(text):
    """
    Extract skills from a single text using SkillNER pipeline.
    
    Parameters
    ----------
    text : str
        Text to extract skills from
    
    Returns
    -------
    list
        List of extracted skill strings
    """
    if pd.isna(text) or text == '':
        return []
    
    try:
        # Create document
        doc = Document()
        
        # Create extraction pipeline
        pipeline = Pipeline()
        
        # Add text loader
        pipeline.add_node(
            StrTextLoader(str(text)),
            name='loader'
        )
        
        # Add skill matcher
        pipeline.add_node(
            SlidingWindowMatcher(
                query_skills,
                max_window_size=MAX_WINDOW_SIZE,
                pre_filter=lambda word: word.lower()
            ),
            name='matcher'
        )
        
        # Add conflict resolver
        pipeline.add_node(
            SpanProcessor(
                dict_filters={
                    "max_candidate": lambda span: max(span.li_candidates, key=len)
                }
            ),
            name="conflict_resolver"
        )
        
        # Run pipeline
        pipeline.run(doc)
        
        # Extract skill strings
        skills = []
        for sentence in doc:
            for span in sentence.li_spans:
                if 'max_candidate' in span.metadata:
                    max_candidate = span.metadata['max_candidate']
                    skill_text = " ".join(sentence[max_candidate.window])
                    skills.append(skill_text)
        
        return skills
        
    except Exception as e:
        # Silently handle errors
        return []

def process_text_batch(texts):
    """Process a batch of texts"""
    results = []
    for text in texts:
        skills = extract_skills_from_text(text)
        results.append({
            'skills': skills,
            'num_skills': len(skills)
        })
    return results

def process_parquet_file_in_chunks(filepath, chunk_size=ROWS_PER_CHUNK):
    """
    Process a single parquet file in chunks.
    """
    try:
        pf = pd.read_parquet(filepath)
    except Exception as e:
        print(f"Error reading {filepath.name}: {e}")
        return None
    
    total_rows = len(pf)
    all_results = []
    
    for start_idx in range(0, total_rows, chunk_size):
        end_idx = min(start_idx + chunk_size, total_rows)
        chunk = pf.iloc[start_idx:end_idx].copy()
        
        # Extract skills
        texts = chunk[TEXT_COLUMN].tolist()
        skill_results = process_text_batch(texts)
        
        # Add results
        chunk['extracted_skills'] = [r['skills'] for r in skill_results]
        chunk['num_skills'] = [r['num_skills'] for r in skill_results]
        
        all_results.append(chunk)
        
        # Clear memory
        del chunk, texts, skill_results
    
    result_df = pd.concat(all_results, ignore_index=True)
    
    del pf, all_results
    gc.collect()
    
    return result_df

def process_single_file(filepath):
    """Process a single file"""
    try:
        result_df = process_parquet_file_in_chunks(filepath)
        
        if result_df is None:
            return (filepath, False, "Failed to read file")
        
        # Save result
        output_path = OUTPUT_DIR / f"{filepath.stem}_processed.parquet"
        result_df.to_parquet(output_path, index=False)
        
        # Mark as processed
        mark_file_as_processed(filepath.name)
        
        message = f"Processed {len(result_df)} rows, {result_df['num_skills'].sum()} skills"
        
        del result_df
        gc.collect()
        
        return (filepath, True, message)
        
    except Exception as e:
        return (filepath, False, str(e))

print("✓ Helper functions loaded")

## Test Extraction on Sample Text

In [ ]:
# Test with sample text
sample_text = """
We are looking for a Software Engineer with experience in Python, Java, and SQL.
Required skills: Machine Learning, Data Analysis, Cloud Computing (AWS, Azure).
Nice to have: Docker, Kubernetes, CI/CD.
"""

print("Testing skill extraction...")
print(f"\nSample text: {sample_text}")

extracted_skills = extract_skills_from_text(sample_text)

print(f"\n✓ Extracted {len(extracted_skills)} skills:")
for skill in extracted_skills:
    print(f"  - {skill}")

if len(extracted_skills) == 0:
    print("\n⚠️  Warning: No skills extracted!")
    print("This might be normal if the sample text doesn't match the knowledge base.")
else:
    print("\n✓ Skill extraction is working!")

## Discover Parquet Files

In [ ]:
# Get all parquet files
all_parquet_files = sorted(INPUT_DIR.glob("*.parquet"))
print(f"Found {len(all_parquet_files)} parquet files in {INPUT_DIR}")

if len(all_parquet_files) == 0:
    raise FileNotFoundError(f"No parquet files found in {INPUT_DIR}")

# Get total size
total_size = sum(f.stat().st_size for f in all_parquet_files)
print(f"Total size: {total_size / (1024**3):.2f} GB")

# Filter processed files
if RESUME_FROM_CHECKPOINT:
    processed_files = get_processed_files()
    files_to_process = [f for f in all_parquet_files if f.name not in processed_files]
    print(f"Already processed: {len(processed_files)} files")
    print(f"Remaining: {len(files_to_process)} files")
else:
    files_to_process = all_parquet_files
    print(f"Will process all {len(files_to_process)} files")

# Show first few files
print("\nFirst 5 files to process:")
for f in files_to_process[:5]:
    size_mb = f.stat().st_size / (1024**2)
    print(f"  - {f.name} ({size_mb:.2f} MB)")

## Process Files

**Choose one of the following cells to run:**
- **Sequential Processing** (safer, slower)
- **Parallel Processing** (faster, uses more memory)

In [ ]:
# OPTION 1: SEQUENTIAL PROCESSING (Recommended for first run)

import time

total_batches = (len(files_to_process) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\nProcessing {len(files_to_process)} files in {total_batches} batches (SEQUENTIAL)")
print("="*70)

overall_start = time.time()

for batch_idx in range(total_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(files_to_process))
    batch_files = files_to_process[start_idx:end_idx]
    
    print(f"\nBatch {batch_idx + 1}/{total_batches}: Files {start_idx + 1}-{end_idx}")
    print("-"*70)
    
    batch_start = time.time()
    
    # Process files sequentially
    for filepath in tqdm(batch_files, desc="Processing"):
        filepath_obj, success, message = process_single_file(filepath)
        if success:
            print(f"✓ {filepath.name}: {message}")
        else:
            print(f"✗ {filepath.name}: {message}")
    
    batch_elapsed = time.time() - batch_start
    gc.collect()
    
    print(f"\n✓ Batch completed in {batch_elapsed:.2f} seconds")
    
    # Estimate remaining time
    if batch_idx < total_batches - 1:
        avg_time = (time.time() - overall_start) / (batch_idx + 1)
        remaining = avg_time * (total_batches - batch_idx - 1)
        print(f"  Estimated remaining: {remaining/60:.1f} minutes")
    
    print("="*70)

total_elapsed = time.time() - overall_start
print(f"\n✓ ALL COMPLETE in {total_elapsed/60:.2f} minutes!")

In [ ]:
# OPTION 2: PARALLEL PROCESSING (Faster but uses more memory)

if USE_PARALLEL:
    from joblib import Parallel, delayed
    import time
    
    total_batches = (len(files_to_process) + BATCH_SIZE - 1) // BATCH_SIZE
    
    print(f"\nProcessing {len(files_to_process)} files in {total_batches} batches (PARALLEL)")
    print(f"Using {N_WORKERS} workers")
    print("="*70)
    
    overall_start = time.time()
    
    for batch_idx in range(total_batches):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min(start_idx + BATCH_SIZE, len(files_to_process))
        batch_files = files_to_process[start_idx:end_idx]
        
        print(f"\nBatch {batch_idx + 1}/{total_batches}: Files {start_idx + 1}-{end_idx}")
        print("-"*70)
        
        batch_start = time.time()
        
        # Process in parallel
        results = Parallel(n_jobs=N_WORKERS, verbose=0)(
            delayed(process_single_file)(filepath)
            for filepath in batch_files
        )
        
        # Report results
        success_count = 0
        for filepath, success, message in results:
            if success:
                print(f"✓ {filepath.name}: {message}")
                success_count += 1
            else:
                print(f"✗ {filepath.name}: {message}")
        
        batch_elapsed = time.time() - batch_start
        gc.collect()
        
        print(f"\n✓ Batch completed: {success_count}/{len(batch_files)} files in {batch_elapsed:.2f} seconds")
        
        if batch_idx < total_batches - 1:
            avg_time = (time.time() - overall_start) / (batch_idx + 1)
            remaining = avg_time * (total_batches - batch_idx - 1)
            print(f"  Estimated remaining: {remaining/60:.1f} minutes")
        
        print("="*70)
    
    total_elapsed = time.time() - overall_start
    print(f"\n✓ ALL COMPLETE in {total_elapsed/60:.2f} minutes!")
else:
    print("Parallel processing is disabled. Use the sequential cell above.")

## Summary Statistics

In [ ]:
# Get all processed output files
output_files = sorted(OUTPUT_DIR.glob("*_processed.parquet"))

print(f"Total output files: {len(output_files)}")

# Calculate statistics
total_rows = 0
total_skills = 0

print("\nCalculating statistics...")
for f in tqdm(output_files, desc="Reading files"):
    df = pd.read_parquet(f, columns=['num_skills'])
    total_rows += len(df)
    total_skills += df['num_skills'].sum()

avg_skills = total_skills / total_rows if total_rows > 0 else 0

print("\n" + "="*70)
print("FINAL STATISTICS")
print("="*70)
print(f"Files processed: {len(output_files)}")
print(f"Total rows: {total_rows:,}")
print(f"Total skills: {total_skills:,}")
print(f"Average skills/row: {avg_skills:.2f}")
print("="*70)

## Inspect Results

In [ ]:
# Load one example file
if len(output_files) > 0:
    example_file = output_files[0]
    df_example = pd.read_parquet(example_file)
    
    print(f"Example file: {example_file.name}")
    print(f"Shape: {df_example.shape}")
    print(f"\nColumns: {df_example.columns.tolist()}")
    
    # Show rows with skills
    df_with_skills = df_example[df_example['num_skills'] > 0]
    
    if len(df_with_skills) > 0:
        print(f"\nRows with skills: {len(df_with_skills)} / {len(df_example)} ({100*len(df_with_skills)/len(df_example):.1f}%)")
        print("\nExamples:")
        for idx, row in df_with_skills.head(3).iterrows():
            print(f"\nRow {idx}:")
            print(f"  Skills ({row['num_skills']}): {row['extracted_skills']}")
    else:
        print("\n⚠️  No skills found in this file")
        print("This might indicate:")
        print("  - Knowledge base doesn't match your JD content")
        print("  - Wrong text column selected")
        print("  - JDs are in different language")
else:
    print("No output files found")